# Proyecto semestral PMD — Red de buses Gran Concepción

**Procesamiento Masivo de Datos** · USM 2026-2 · Prof. Diego Ramírez · Paralelo 701
**Integrantes:** Javier Fernández · Benjamín Medina

## De qué trata este proyecto

Queremos responder una pregunta simple pero con datos reales y masivos: **¿cómo se distribuye
y se comporta la red de buses del Gran Concepción, tanto en su trazado declarado como en su
operación real en vivo?**

Para eso combinamos dos familias de la asignatura sobre la misma fuente de datos (la API no
oficial de TransApp/RED que documenta la red del Gran Concepción):

| Familia | Qué datos | Dónde vive |
|---|---|---|
| **Geoespacial (PostGIS)** | GTFS estático: 2.256 paraderos, 190 rutas (560 variantes), ~19.000 puntos de trazado | Este notebook, contra Postgres local |
| **Streaming (Redis)** | Telemetría en vivo: posición, patente y rumbo de cada bus activo, recolectada cada 10 min | Upstash Redis, alimentado por `collect.py` vía GitHub Actions |

Este notebook cubre la parte **PostGIS**: carga, modelo de datos y las consultas espaciales
sobre el trazado real de la red.

## Los datos

| Fuente | Qué es | Cómo se obtuvo |
|---|---|---|
| `stop.csv` | Paraderos (parada, nombre, rutas que la usan, coordenadas) | Export del feed estático de la zona del Gran Concepción (`public_id f083f24e-7fee-4fdc-9520-dbbdae877f07`) |
| `route.csv` | Rutas con su dirección (`I`=Ida, `R`=Retorno) y metadatos | Mismo feed |
| `shape.csv` | Secuencia de puntos que forman el trazado real de cada ruta+dirección | Mismo feed |

Todo se carga a PostGIS en **SRID 5361** (SIRGAS-Chile / UTM 19S), el sistema métrico que usa
el curso, con `cargar.py` (fuera de este notebook — correr primero `docker compose up -d --wait`
y `python cargar.py`).

## Modelo de datos

- **`paraderos`** — un punto (`GEOMETRY(Point, 5361)`) por parada, con índice GiST.
- **`rutas_trazado`** — una línea (`GEOMETRY(LineString, 5361)`) por combinación ruta+dirección
  (`shape_id`), construida ordenando los puntos de `shape.csv` por su `sequence`, con índice GiST.

## Cómo correr este notebook

1. `docker compose up -d --wait` y `python cargar.py` (deben terminar en `VALIDACIÓN OK`).
2. Abrir este notebook y ejecutar **Run All**.
3. Guardarlo con las salidas antes de subirlo al repositorio — así se ve sin tener que volver
   a correr nada.

## 0. Conexión y set up

In [11]:
import psycopg2
import pandas as pd

DB = dict(host="localhost", port=5432, dbname="transporte", user="pmd", password="pmd_pass")
conn = psycopg2.connect(**DB)
pd.set_option("display.max_rows", 20)
print("Conectado a PostGIS.")

Conectado a PostGIS.


## 1. Cuántos paraderos y rutas quedaron cargados

Primer chequeo: que los conteos coincidan con los archivos fuente, antes de analizar nada.

In [12]:
conteos = pd.read_sql('''
    SELECT 'paraderos' AS tabla, COUNT(*) AS filas FROM paraderos
    UNION ALL
    SELECT 'rutas_trazado', COUNT(*) FROM rutas_trazado
''', conn)
conteos

C:\Users\benja\AppData\Local\Temp\ipykernel_21644\2158996601.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  conteos = pd.read_sql('''


,tabla,filas
0,paraderos,2256
1,rutas_trazado,198


**Qué significa.** `paraderos` debería tener 2.256 filas (una por cada fila de `stop.csv`
con coordenadas válidas) y `rutas_trazado` 198 (las combinaciones ruta+dirección que sí tienen
un trazado real en `shape.csv` — no todas las 190 rutas tienen shape para ambos sentidos).

## 2. Pregunta 1 — Paraderos cercanos a un punto dado

**Qué hacemos.** Dado un punto (ejemplo: el centro de Concepción), buscamos todos los
paraderos a 300 m o menos, usando `ST_DWithin` sobre el índice espacial — no comparando
contra los 2.256 paraderos uno por uno.

In [13]:
Q_CERCANOS = '''
SELECT stop_id, name, ROUND(ST_Distance(paraderos.geom, pt.geom)::numeric, 1) AS distancia_m
FROM paraderos,
     (SELECT ST_Transform(ST_SetSRID(ST_MakePoint(-73.0503, -36.8270), 4326), 5361) AS geom) pt
WHERE ST_DWithin(paraderos.geom, pt.geom, 300)
ORDER BY distancia_m;
'''
df_cercanos = pd.read_sql(Q_CERCANOS, conn)
df_cercanos

C:\Users\benja\AppData\Local\Temp\ipykernel_21644\2627964948.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_cercanos = pd.read_sql(Q_CERCANOS, conn)


,stop_id,name,distancia_m
0,P37520,Correos De Chile,146.3
1,P39496,Freire Esq. Caupolicán,183.8
2,P40998,Rengo Esq O'higgins,192.3
3,P42319,San Martín Esq. Anibal Pinto,206.6
4,P49394,Rengo Esq. Barros Arana,207.6
5,P39625,Freire Esq Anibal Pinto,208.9
6,P42318,San Martín Esq. Caupolicán,229.3
7,P45069,Ex Ymca,243.4
8,P39627,Freire Esq Rengo,243.7
9,P37480,O'higgins Esq. Rengo,243.9


**Resultado y qué significa.** Estos son los paraderos que un pasajero parado en ese punto
podría alcanzar caminando (≤300 m). Sirve como ejemplo del patrón "punto de interés → paraderos
cercanos", la base de cualquier análisis de cobertura peatonal.

## 3. Pregunta 2 — Las 10 rutas con trazado más largo

**Qué hacemos.** `ST_Length` sobre cada `LineString` ya en SRID 5361 (métrico), así que el
resultado es directamente en metros, sin conversión adicional.

In [14]:
Q_LARGAS = '''
SELECT route_name, direction, ROUND(ST_Length(geom)::numeric) AS largo_m
FROM rutas_trazado
ORDER BY largo_m DESC
LIMIT 10;
'''
df_largas = pd.read_sql(Q_LARGAS, conn)
df_largas

C:\Users\benja\AppData\Local\Temp\ipykernel_21644\1603455463.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_largas = pd.read_sql(Q_LARGAS, conn)


,route_name,direction,largo_m
0,201AU,I,68789.0
1,201AU,R,66619.0
2,201,I,66421.0
3,201,R,63946.0
4,10C,I,49648.0
5,80J,I,49326.0
6,10P,R,49284.0
7,80Q,R,48458.0
8,304,R,48150.0
9,302,R,48150.0


**Resultado y qué significa.** Las rutas `201` / `201AU` encabezan con más de 60 km de
trazado — son las rutas troncales de mayor recorrido de la red. Esto da una primera noción de
qué rutas cubren más territorio, antes de cruzar con tiempos de viaje o frecuencia.

## 4. Pregunta 3 — Paraderos que cubre una ruta específica

**Qué hacemos.** Cruce ruta↔paradero: qué paraderos están a 50 m o menos del trazado real
de una ruta dada. Útil para validar que el trazado declarado efectivamente pasa cerca de los
paraderos que dice servir.

**Nota sobre la ruta elegida.** No todas las rutas tienen `shape` para ambos sentidos: `12A`
solo tiene trazado en sentido **R** (Retorno) y no existe `12A_I` en los datos fuente (`shape.csv`).
Esto no es un error del pipeline — es así en la fuente — pero hay que saberlo antes de filtrar,
si no la consulta devuelve 0 filas sin explicación. Usamos `12A` / `R`, el sentido que sí existe.


In [15]:
Q_COBERTURA_RUTA = '''
SELECT DISTINCT p.stop_id, p.name
FROM paraderos p
JOIN rutas_trazado r ON ST_DWithin(p.geom, r.geom, 50)
WHERE r.route_name = '12A' AND r.direction = 'R'
ORDER BY p.name;
'''
df_cobertura = pd.read_sql(Q_COBERTURA_RUTA, conn)
print(f"Paraderos a <=50 m del trazado de la ruta 12A (Retorno): {len(df_cobertura)}")
df_cobertura

Paraderos a <=50 m del trazado de la ruta 12A (Retorno): 210


C:\Users\benja\AppData\Local\Temp\ipykernel_21644\3946075406.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_cobertura = pd.read_sql(Q_COBERTURA_RUTA, conn)


,stop_id,name
0,P35694,21 De Mayo Esq Abraham Valenzuela
1,P39637,21 De Mayo Esq Argomedo Oriente
2,P35696,21 De Mayo Esq Argomedo Poniente
3,P50066,21 De Mayo Esq Capitan Orellana
4,P39641,21 De Mayo Esq Cuarto Centenario Norte
...,...,...
205,P20554362,Terminal
206,PPV420002,Terminal B02
207,P38560,Valdivia Esq Colon
208,P39643,Veintiuno De Mayo Esq Pto Saavedra


**Resultado y qué significa.** Esta cantidad de paraderos "cubiertos" por el trazado real
es la que compararíamos contra los paraderos que `stop.csv` dice que sirve la ruta (columna
`routes`) para detectar diferencias entre lo declarado y lo geométricamente real.

## 5. Plan de ejecución — ¿se está usando el índice espacial?

**Qué hacemos.** Corremos `EXPLAIN ANALYZE` sobre la consulta de la Pregunta 1 para confirmar
qué índice usa el motor y en qué orden.

In [16]:
with conn.cursor() as cur:
    cur.execute("EXPLAIN ANALYZE " + Q_CERCANOS)
    plan = "\n".join(row[0] for row in cur.fetchall())
print(plan)

Sort  (cost=259.02..259.02 rows=1 width=67) (actual time=0.079..0.080 rows=14 loops=1)
  Sort Key: (round((st_distance(paraderos.geom, '0101000020F1140000225AFE9D00EF00410CE33855FC915641'::geometry))::numeric, 1))
  Sort Method: quicksort  Memory: 25kB
  ->  Bitmap Heap Scan on paraderos  (cost=4.40..259.01 rows=1 width=67) (actual time=0.038..0.070 rows=14 loops=1)
        Filter: st_dwithin(geom, '0101000020F1140000225AFE9D00EF00410CE33855FC915641'::geometry, '300'::double precision)
        Rows Removed by Filter: 1
        Heap Blocks: exact=12
        ->  Bitmap Index Scan on idx_paraderos_geom  (cost=0.00..4.40 rows=17 width=0) (actual time=0.021..0.021 rows=15 loops=1)
              Index Cond: (geom && st_expand('0101000020F1140000225AFE9D00EF00410CE33855FC915641'::geometry, '300'::double precision))
Planning Time: 0.148 ms
Execution Time: 0.106 ms


**Qué significa.** El plan muestra un `Bitmap Index Scan` sobre `idx_paraderos_geom`
(el índice GiST de `paraderos`): primero acota con el índice a los paraderos cuyo *bounding box*
está cerca del punto, y solo sobre esos calcula la distancia exacta con `ST_Distance`. Sin este
índice, la consulta tendría que comparar el punto contra los 2.256 paraderos uno por uno.

## 6. Cierre

Con esto queda demostrado el pilar **Geoespacial / PostGIS** del proyecto: datos reales
cargados con integridad verificada, modelo de datos en el SRID correcto, consultas espaciales
funcionando con los patrones pedidos por el curso (`ST_DWithin`, `ST_Length`, join espacial), y
evidencia de que los índices GiST se usan de verdad.

La parte **Streaming (Redis)** vive fuera de este notebook — ver `collect.py` y el workflow de
GitHub Actions — porque corre de forma continua, no en una sola ejecución.